In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import json

PROJECT = Path(
    "/content/drive/MyDrive/Projects/humanoid-challenge"
)

VIDEO = PROJECT / "data/raw/pilot_pick_place_01.mov"
assert VIDEO.exists(), f"Video not found: {VIDEO}"

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), "OpenCV could not open the video"

frames = []
timestamps_s = []

while True:
    ok, frame = cap.read()
    if not ok:
        break

    timestamp_s = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

    # Reduce display size while preserving the aspect ratio.
    height, width = frame.shape[:2]
    scale = min(1.0, 720 / max(height, width))

    small_frame = cv2.resize(
        frame,
        (round(width * scale), round(height * scale)),
    )

    # OpenCV uses BGR; Matplotlib expects RGB.
    rgb_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    frames.append(rgb_frame)
    timestamps_s.append(float(timestamp_s))

cap.release()

assert frames, "No frames were decoded"
assert all(
    later > earlier
    for earlier, later in zip(timestamps_s, timestamps_s[1:])
), "Timestamps are not increasing; stop here and share this result."

print(f"Loaded {len(frames)} frames")
print(f"Last frame timestamp: {timestamps_s[-1]:.3f} seconds")

In [ ]:
def show_frame(frame_index):
    fig, ax = plt.subplots(figsize=(6, 9))

    ax.imshow(frames[frame_index])
    ax.set_title(
        f"Frame {frame_index} | "
        f"{timestamps_s[frame_index]:.3f} seconds"
    )
    ax.axis("off")

    plt.show()
    plt.close(fig)


slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frames) - 1,
    step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="90%"),
)

viewer = widgets.interactive_output(
    show_frame,
    {"frame_index": slider},
)

display(slider, viewer)

In [ ]:
event_frames = {
    "closure_start": 129,
    "grasp_established": 149,
    "lift_off": 172,
    "placement_contact": 315,
    "opening_start": 340,       # Fill in
    "release_complete": 345,
}

# Optional: record uncertainty as [earliest_frame, latest_frame].
uncertainty_frames = {
    # "grasp_established": [120, 126],
}

notes = (
    "Add any uncertainty here, such as fingers being hidden "
    "or a small adjustment after placement."
)

In [ ]:
assert all(
    isinstance(index, int) and 0 <= index < len(frames)
    for index in event_frames.values()
), "Replace every None with a valid frame number."

ordered_frames = list(event_frames.values())
assert ordered_frames == sorted(ordered_frames), (
    "Check the event order."
)

annotation = {
    "video": str(VIDEO.relative_to(PROJECT)),
    "method": "manual visual annotation",
    "frame_index_convention": "zero-based decoded frame index",
    "events": {
        name: {
            "frame": index,
            "time_s": timestamps_s[index],
        }
        for name, index in event_frames.items()
    },
    "uncertainty_frames": uncertainty_frames,
    "notes": notes,
}

output = (
    PROJECT / "data/processed/pilot_01/human_events.json"
)
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(annotation, indent=2))

print(json.dumps(annotation["events"], indent=2))
print(f"\nSaved to: {output}")

In [ ]:
import numpy as np

assert all(
    isinstance(frame, int) and 0 <= frame < len(frames)
    for frame in event_frames.values()
), "Fill in both missing frame numbers."

event_times = {
    name: timestamps_s[frame]
    for name, frame in event_frames.items()
}

assert (
    event_times["closure_start"]
    <= event_times["grasp_established"]
    <= event_times["lift_off"]
    <= event_times["placement_contact"]
    <= event_times["opening_start"]
    <= event_times["release_complete"]
), "Review the event order."

# One command every 0.05 seconds: the simulator runs at 20 Hz.
control_times = np.arange(
    0,
    timestamps_s[-1] + 1e-9,
    1 / 20,
)

# Our Panda gripper uses -1 for opening and +1 for closing.
gripper_commands = np.where(
    (control_times >= event_times["closure_start"])
    & (control_times < event_times["opening_start"]),
    1.0,
    -1.0,
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.step(control_times, gripper_commands, where="post")

for name, time_s in event_times.items():
    ax.axvline(time_s, color="gray", alpha=0.4)
    ax.text(
        time_s, 0,
        name.replace("_", " "),
        rotation=90,
        va="center",
        ha="right",
        fontsize=9,
    )

ax.set(
    xlabel="Video time (seconds)",
    ylabel="Gripper command",
    yticks=[-1, 1],
    yticklabels=["Open", "Close"],
    ylim=(-1.3, 1.3),
)
plt.show()

Let's define where the robot should grasp the bowl with respect to the hand marker

In [ ]:
import pandas as pd
import numpy as np

trajectory_path = (
    PROJECT / "data/processed/pilot_01/marker_smoothed.csv"
)

trajectory = pd.read_csv(trajectory_path)

print(trajectory.columns.tolist())
display(trajectory.head())

In [ ]:
position_columns = ["x_m", "y_m", "height_m"]

sample_times = trajectory["seconds"].to_numpy()

assert np.all(np.diff(sample_times) > 0), (
    "Trajectory timestamps must increase."
)

event_positions = {}

for name, time_s in event_times.items():
    assert sample_times[0] <= time_s <= sample_times[-1], (
        f"{name} is outside the tracked interval."
    )

    position = np.array([
        np.interp(
            time_s,
            sample_times,
            trajectory[column].to_numpy(),
        )
        for column in position_columns
    ])

    event_positions[name] = position

event_table = pd.DataFrame(
    event_positions,
    index=position_columns,
).T

display(event_table.rename(
    columns={column: column.replace("_m", "_mm")
             for column in position_columns}
) * 1000)

In [ ]:
grasp_position = event_positions["grasp_established"]

relative_positions = event_table - grasp_position

print("Marker displacement relative to grasp, in millimetres:")
display(relative_positions * 1000)